# Cross-Encoder VAL · 리뷰 5개를 3개와 2개로 나눴을 때의 순위

같은 쿼리와 같은 후보 상품에서 **리뷰 5개를 한 번에 읽는 기준선(Full 5)**과 **3개·2개로 나눠 읽는 네 방식**을 비교한다. 직전 실험과 같은 ColBERT epoch 5의 상위 100개 후보, Mixed Cross-Encoder epoch 3, 입력 쌍당 448토큰, 리뷰당 최대 58토큰을 사용한다. 모델 가중치는 고정한다.

기존 SageMaker 프로젝트에서 `conda_pytorch` 커널로 **Run All**을 실행한다. 기본값은 직전 VAL 350개다. 짧게 점검하려면 첫 코드 셀의 `VAL_LIMIT = 10`으로 바꾼다. 필요한 입력은 기존 학습 실행의 가중치, 설정, VAL 후보 캐시와 프로젝트 데이터다.

In [ ]:
from contextlib import nullcontext
from pathlib import Path
from tempfile import TemporaryDirectory
import hashlib
import json

import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import display
from torch import nn
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

PROJECT_DIR = Path('/home/ec2-user/SageMaker/20262R0136DATA30400')
VAL_LIMIT = 350          # None이면 해당 Mixed 실행에 저장된 VAL 전체
MIXED_RUN_ID = None      # 조건이 같은 학습 실행이 여럿이면 폴더명 지정
INFER_BATCH_SIZE = None  # 기본값은 기존 cross_infer_batch. GPU 메모리에 맞춰 줄일 수 있다.

RUN_DIR = PROJECT_DIR / 'data304_outputs'
MIXED_ROOT = RUN_DIR / 'cross_hard4_mixed4_lr5e-6'
COLBERT_EPOCH, CROSS_EPOCH = 5, 3
SEARCH = dict(ann_per_token=1024, ann_nprobe=32, ann_pool=900, candidate_topk=100)
METHODS = ('Full 5', 'Score Mean', 'Score Max', 'Score LogSumExp', 'Embedding Mean')
REVIEW_GROUPS = {'1-5': slice(0, 5), '1-3': slice(0, 3), '4-5': slice(3, 5)}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 1. 같은 검증 쿼리와 후보 불러오기

직전 검증과 같은 검색 설정(토큰당 이웃 1,024개, `nprobe=32`, ANN 풀 900개)으로 저장된 후보를 원래 MaxSim 순서로 읽는다. 다섯 방식 모두 같은 배열에서 시작한다. 정답이 후보에 없는 쿼리도 전체 성능 지표의 분모에 들어간다.

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8-sig'))


def signature(value):
    encoded = json.dumps(value, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(encoded).hexdigest()


def file_sha(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


def find_mixed_run():
    parent = MIXED_ROOT / f'colbert_epoch_{COLBERT_EPOCH:02d}'
    manifests = ([parent / MIXED_RUN_ID / 'settings.json'] if MIXED_RUN_ID is not None
                 else sorted(parent.glob('*/settings.json')))
    matches = []
    for path in manifests:
        if not path.is_file():
            continue
        settings = read_json(path)
        training, mining = settings['training'], settings['mining']
        epoch_dir = path.parent / f'epoch_{CROSS_EPOCH:02d}'
        weights = epoch_dir / 'checkpoints' / 'cross_mixed.pt'
        if (weights.is_file()
                and training['colbert_epoch'] == COLBERT_EPOCH
                and CROSS_EPOCH in training['target_epochs']
                and (training['hard_count'], training['original_count'], training['original_mode']) == (4, 4, 'mixed')
                and training['lr'] == 5e-6
                and (training['config']['cross_tokens'], training['config']['cross_tokens_per_review']) == (448, 58)
                and all(mining[key] == value for key, value in SEARCH.items())
                and training['mining'] == signature(mining)):
            matches.append((path.parent, settings, weights))
    if not matches:
        raise FileNotFoundError(f'Mixed epoch 3의 settings.json과 epoch_03/checkpoints/cross_mixed.pt를 확인하세요. {parent}')
    if len(matches) > 1:
        names = [folder.name for folder, _, _ in matches]
        raise ValueError(f'조건이 같은 실행이 여러 개입니다. 첫 셀의 MIXED_RUN_ID에 폴더명을 지정하세요. {names}')
    return matches[0]


SOURCE_RUN, SAVED, CROSS_WEIGHTS = find_mixed_run()
MINING = SAVED['mining']
TRAIN_CONFIG = SAVED['training']['config']
TRAIN_LENGTH = TRAIN_CONFIG['cross_tokens']
PER_REVIEW = TRAIN_CONFIG['cross_tokens_per_review']
CATALOG_PATH = PROJECT_DIR / 'catalog.json'
GOLD_PATH = PROJECT_DIR / 'train_gold.json'
SPLIT_PATH = RUN_DIR / 'split.json'
for path, expected_hash in ((CATALOG_PATH, MINING['index']['catalog']),
                            (GOLD_PATH, MINING['gold']), (SPLIT_PATH, MINING['split'])):
    if file_sha(path) != expected_hash:
        raise ValueError(f'후보를 만들 때 사용한 파일과 내용이 다릅니다. {path}')

project_config = dict(TRAIN_CONFIG)
for config_path in (RUN_DIR / 'aspect_vocabulary_and_config.json', RUN_DIR / 'run_manifest.json'):
    if config_path.is_file():
        project_config.update(read_json(config_path)['config'])
BATCH = project_config.get('cross_infer_batch') if INFER_BATCH_SIZE is None else INFER_BATCH_SIZE
if type(BATCH) is not int or BATCH < 1:
    raise ValueError('기존 cross_infer_batch를 확인하거나 첫 셀에서 INFER_BATCH_SIZE에 양의 정수를 지정하세요.')

saved_ids = SAVED['val_query_ids']
split_ids = read_json(SPLIT_PATH)['val_query_ids']
if not saved_ids or saved_ids != split_ids[:len(saved_ids)] or len(set(saved_ids)) != len(saved_ids):
    raise ValueError('저장한 Mixed VAL의 쿼리와 split.json의 순서를 확인하세요.')
if VAL_LIMIT is not None and (type(VAL_LIMIT) is not int or not 1 <= VAL_LIMIT <= len(saved_ids)):
    raise ValueError(f'VAL_LIMIT은 1~{len(saved_ids)} 또는 None으로 지정하세요.')
VAL_IDS = saved_ids[:VAL_LIMIT]
gold_rows = {r['query_id']: r for r in read_json(GOLD_PATH)}
VAL = [gold_rows[qid] for qid in VAL_IDS]
del gold_rows


def metadata_text(item):
    fields = [('title', 'Product'), ('store', 'Store'), ('features', 'Features'), ('description', 'Description')]
    parts = [f'{label}: {item[key].strip()}' for key, label in fields if item[key].strip()]
    return ' [SEP] '.join(parts) or 'Product metadata not available'


catalog = read_json(CATALOG_PATH)
ITEM_IDS = [item['item_id'] for item in catalog]
ITEM_TO_IDX = {iid: i for i, iid in enumerate(ITEM_IDS)}
CAT_TEXTS = [metadata_text(item) for item in catalog]
del catalog
if len(ITEM_IDS) != len(ITEM_TO_IDX) or any(row['item_id'] not in ITEM_TO_IDX for row in VAL):
    raise ValueError('카탈로그의 상품 ID와 VAL 정답을 확인하세요.')

In [ ]:
def load_val_candidates():
    mining_key = signature(MINING)[:20]
    split_key = 'val_' + signature(saved_ids)[:16]  # 일부만 평가해도 원래 350건 캐시를 읽는다.
    roots = []
    for field, fallback in [('candidate_cache', MIXED_ROOT), ('previous_readonly_candidates', RUN_DIR / 'cross_hard_negative')]:
        recorded = Path(SAVED[field])
        if recorded.name != mining_key:
            raise ValueError(f'검색 설정과 후보 캐시의 식별자가 다릅니다. {recorded}')
        # 프로젝트 폴더를 옮겼을 때도 새 경로에서 같은 캐시를 찾는다.
        roots.extend([fallback / 'candidate_cache' / mining_key / split_key, recorded / split_key])
    roots = list(dict.fromkeys(roots))
    shards = {}
    for root in roots:
        for shard in sorted(root.glob('*.npz')):
            shards.setdefault(shard.name, shard)  # 원래 학습 코드도 mixed 캐시를 우선한다.

    n = len(VAL_IDS)
    topk = min(SEARCH['candidate_topk'], len(ITEM_IDS))
    candidates = np.empty((n, topk), dtype=np.int32)
    ranks = np.empty(n, dtype=np.int32)
    filled = np.zeros(n, dtype=bool)
    for path in shards.values():
        start, end = map(int, path.stem.split('_'))
        if not 0 <= start < end <= len(saved_ids):
            raise ValueError(f'VAL 범위가 잘못된 후보 묶음입니다. {path}')
        if start >= n:
            continue
        stop = min(n, end)
        with np.load(path, allow_pickle=False) as shard:
            docs, ann = shard['candidates'], shard['ann_ranks']
            if (not np.array_equal(shard['query_ids'], np.asarray(saved_ids[start:end], dtype=str))
                    or docs.shape != (end - start, topk) or ann.shape != (end - start,)
                    or docs.dtype.kind not in 'iu' or ann.dtype.kind not in 'iu'
                    or np.any((docs < 0) | (docs >= len(ITEM_IDS)))
                    or np.any((ann < 0) | (ann > SEARCH['ann_pool']))):
                raise ValueError(f'쿼리 순서, 후보 또는 ANN 순위가 잘못되었습니다. {path}')
            docs, ann = docs[:stop - start], ann[:stop - start]
            occupied = filled[start:stop]
            if (np.any(candidates[start:stop][occupied] != docs[occupied])
                    or np.any(ranks[start:stop][occupied] != ann[occupied])):
                raise ValueError(f'같은 쿼리에 서로 다른 후보가 저장되어 있습니다. {path}')
            candidates[start:stop], ranks[start:stop] = docs, ann
            filled[start:stop] = True
    if not filled.all():
        i = int(np.flatnonzero(~filled)[0])
        raise FileNotFoundError(f'기존 Mixed VAL 후보에서 {VAL_IDS[i]}가 누락되었습니다. npz 폴더: {roots}')
    if np.any(np.diff(np.sort(candidates, axis=1), axis=1) == 0):
        raise ValueError('한 쿼리의 상위 후보에 중복 상품이 있습니다.')
    return candidates, ranks


CANDIDATES, ANN_RANKS = load_val_candidates()
print(f'Cross 가중치 {CROSS_WEIGHTS}\nVAL {len(VAL)}건 × 후보 {CANDIDATES.shape[1]}개 | {SEARCH}')

## 2. 한 후보 상품에 대해 다섯 점수를 계산하는 방법

리뷰 순서를 유지해 A에는 **Review 1–3**, B에는 **Review 4–5**를 넣고, 양쪽에 같은 후보 상품의 메타데이터를 붙인다. 리뷰별 58토큰 상한과 `longest_first`를 세 입력(Full 5, A, B)에 똑같이 적용한다. 분할로 달라지는 것은 448토큰에 맞추는 단계다. 리뷰 한 개가 58토큰을 넘어 처음에 잘린 부분은 세 입력 모두에서 제외된다.

$h_A,h_B$는 각 리뷰 묶음과 **해당 후보 상품을 함께 인코딩한 마지막 층의 `[CLS]`**다. $f$는 학습된 DistilBERT 분류 헤드이고, $s_A=f(h_A)$, $s_B=f(h_B)$는 sigmoid를 적용하기 전의 점수(logit)다.

| 방식 | 후보 상품의 최종 점수 |
|---|---|
| Full 5 | 리뷰 1–5와 상품을 함께 인코딩한 기존 Cross-Encoder 점수 |
| Score Mean | $(s_A+s_B)/2$ |
| Score Max | $\max(s_A,s_B)$ |
| Score LogSumExp | $\log(\exp(s_A)+\exp(s_B))$, 온도 $T=1$ |
| Embedding Mean | $f((h_A+h_B)/2)$ |

두 평균은 묶음 A와 B에 각각 ½의 가중치를 준다. Embedding Mean은 `pre_classifier`와 ReLU **앞에서** 평균내므로 Score Mean과 다른 순위를 낼 수 있다. 분할 방식 네 가지가 A와 B의 인코딩 결과를 공유한다.

In [ ]:
backbone = TRAIN_CONFIG['backbone']
config = AutoConfig.from_pretrained(backbone)
config.num_labels = 1
TOKENIZER = AutoTokenizer.from_pretrained(backbone, use_fast=True)
if config.model_type != 'distilbert' or not TOKENIZER.is_fast:
    raise ValueError('이 실험은 기존 DistilBERT 분류 헤드와 fast tokenizer를 사용합니다.')
if TRAIN_LENGTH > min(config.max_position_embeddings, TOKENIZER.model_max_length):
    raise ValueError('체크포인트의 입력 길이가 모델 또는 토크나이저의 상한보다 큽니다.')
tokenizer_hash = hashlib.sha256(TOKENIZER.backend_tokenizer.to_str().encode()).hexdigest()
PAIR_SPECIALS = TOKENIZER.num_special_tokens_to_add(pair=True)


def token_lengths(texts):
    return TOKENIZER(texts, add_special_tokens=False, truncation=False, return_length=True,
                     return_attention_mask=False, return_token_type_ids=False, verbose=False)['length']


def prepare_reviews(row):
    reviews = [r['text'].strip() for r in row['reviews']]
    if len(reviews) != 5:
        raise ValueError(f"리뷰가 다섯 개여야 합니다. {row['query_id']}")
    raw_lengths = token_lengths(reviews)
    chunks = []
    for number, review in enumerate(reviews, start=1):
        ids = TOKENIZER.encode(review, add_special_tokens=False, truncation=True, max_length=PER_REVIEW)
        chunks.append(f'Review {number}: ' + TOKENIZER.decode(ids, skip_special_tokens=True))
    texts = [' [SEP] '.join(chunks[span]) for span in REVIEW_GROUPS.values()]
    lengths = token_lengths(texts)
    groups = {}
    for (key, span), text, length in zip(REVIEW_GROUPS.items(), texts, lengths):
        groups[key] = dict(text=text, length=length, n_reviews=len(chunks[span]),
                           n_capped=sum(n > PER_REVIEW for n in raw_lengths[span]))
    return groups


QUERIES = [prepare_reviews(row) for row in VAL]
doc_lengths = np.zeros(len(ITEM_IDS), dtype=np.int32)
used_docs = np.unique(CANDIDATES)
for start in range(0, len(used_docs), 128):
    docs = used_docs[start:start + 128]
    doc_lengths[docs] = token_lengths([CAT_TEXTS[j] for j in docs])


class CrossReranker(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = AutoModelForSequenceClassification.from_config(config)


cross = CrossReranker()
cross.load_state_dict(torch.load(CROSS_WEIGHTS, map_location='cpu', weights_only=True, mmap=True), strict=True)
net = cross.to(DEVICE).eval().requires_grad_(False).net
print(f'{backbone} | {DEVICE} | pair {TRAIN_LENGTH} | review {PER_REVIEW} | batch {BATCH}')

In [ ]:
def amp_context():
    return torch.autocast('cuda', dtype=torch.bfloat16) if USE_BF16 else nullcontext()


def tokenize_pairs(query_text, docs):
    return TOKENIZER([query_text] * len(docs), [CAT_TEXTS[j] for j in docs], padding=True,
                     truncation='longest_first', max_length=TRAIN_LENGTH, return_tensors='pt')


def device_inputs(encoded):
    return {key: value.to(DEVICE) for key, value in encoded.items()}


def pair_cls(inputs):
    hidden = net.distilbert(**inputs, return_dict=True).last_hidden_state
    return hidden[:, 0, :].clone()  # [batch, hidden_dim]만 다음 묶음까지 보관


def head_score(cls):
    projected = torch.relu(net.pre_classifier(cls))
    return net.classifier(net.dropout(projected)).reshape(-1)


def split_scores(h_a, h_b):
    s_a, s_b = head_score(h_a).float(), head_score(h_b).float()
    h_mean = (h_a.float() + h_b.float()) * 0.5
    return {
        'Score Mean': (s_a + s_b) * 0.5,
        'Score Max': torch.maximum(s_a, s_b),
        'Score LogSumExp': torch.logaddexp(s_a, s_b),
        'Embedding Mean': head_score(h_mean).float(),
    }


# 실제 체크포인트의 한 입력에서 분류 헤드와 기본 forward의 점수를 FP32로 대조한다.
with torch.inference_mode():
    probe = device_inputs(tokenize_pairs(QUERIES[0]['1-3']['text'], CANDIDATES[0, :1]))
    native = net(**probe, return_dict=True).logits.reshape(-1).float()
    from_cls = head_score(pair_cls(probe)).float()
    torch.testing.assert_close(from_cls, native, rtol=1e-4, atol=1e-5)
print('체크포인트의 기본 logit과 [CLS] → 분류 헤드 점수가 일치합니다.')

## 3. 후보를 다시 채점하고 잘린 토큰 세기

상품마다 Full 5, A, B를 각각 한 번 인코딩한다. 각 입력에서 448토큰 상한으로 빠진 리뷰 쪽 토큰과 상품 쪽 토큰을 따로 센다. 리뷰 쪽 토큰 수에는 `Review 1` 같은 번호와 내부 `[SEP]`가 포함된다. 점수가 같으면 원래 ColBERT 후보 순서가 앞선 상품을 먼저 둔다.

In [ ]:
def group_inputs(group, docs, totals):
    encoded = tokenize_pairs(group['text'], docs)
    sides = [encoded.sequence_ids(j) for j in range(len(docs))]
    kept_q = np.array([side.count(0) for side in sides])
    kept_d = np.array([side.count(1) for side in sides])
    lost_q = group['length'] - kept_q
    lost_d = doc_lengths[docs] - kept_d
    full = group['length'] + doc_lengths[docs] + PAIR_SPECIALS
    if (np.any(lost_q < 0) or np.any(lost_d < 0) or
            not np.array_equal(lost_q + lost_d, np.maximum(full - TRAIN_LENGTH, 0))):
        raise ValueError('토크나이저의 입력 길이와 잘림 집계가 일치하지 않습니다.')
    totals['n_pairs'] += len(docs)
    totals['truncated_pairs'] += int(np.count_nonzero(lost_q + lost_d))
    totals['review_side_truncated_pairs'] += int(np.count_nonzero(lost_q))
    totals['metadata_side_truncated_pairs'] += int(np.count_nonzero(lost_d))
    totals['review_tokens_removed'] += int(lost_q.sum())
    totals['metadata_tokens_removed'] += int(lost_d.sum())
    totals['metadata_tokens_kept'] += int(kept_d.sum())
    return device_inputs(encoded)


def rank_methods(score_matrix, ranked, gold):
    if not np.isfinite(score_matrix).all():
        raise ValueError('유한하지 않은 Cross-Encoder 점수가 있습니다.')
    results = {}
    for column, method in enumerate(METHODS):
        ordered = ranked[np.argsort(-score_matrix[:, column], kind='stable')]
        pos = np.flatnonzero(ordered == gold)
        results[method] = {
            'cross_top1_item_id': str(ITEM_IDS[ordered[0]]),
            'cross_gold_rank': int(pos[0]) + 1 if len(pos) else None,
        }
    return results


@torch.inference_mode()
def score_query(i):
    row, groups, ranked = VAL[i], QUERIES[i], CANDIDATES[i]
    gold = ITEM_TO_IDX[row['item_id']]
    gold_pos = np.flatnonzero(ranked == gold)
    totals = {}
    for key, group in groups.items():
        totals[key] = dict(n_reviews=group['n_reviews'], n_capped=group['n_capped'],
                           n_pairs=0, truncated_pairs=0, review_side_truncated_pairs=0,
                           metadata_side_truncated_pairs=0, review_tokens_removed=0,
                           metadata_tokens_removed=0, metadata_tokens_kept=0)
    scores = np.empty((len(ranked), len(METHODS)), dtype=np.float32)
    for start in range(0, len(ranked), BATCH):
        docs = ranked[start:start + BATCH]
        with amp_context():
            inputs = group_inputs(groups['1-5'], docs, totals['1-5'])
            full = net(**inputs, return_dict=True).logits.reshape(-1).float()
            h_a = pair_cls(group_inputs(groups['1-3'], docs, totals['1-3']))
            h_b = pair_cls(group_inputs(groups['4-5'], docs, totals['4-5']))
            batch_scores = {'Full 5': full, **split_scores(h_a, h_b)}
        scores[start:start + len(docs)] = torch.stack([batch_scores[m] for m in METHODS], dim=1).cpu().numpy()
    return {
        'query_id': str(row['query_id']), 'gold_item_id': str(row['item_id']),
        'gold_ann_rank_in_pool': int(ANN_RANKS[i]) or None,
        'gold_maxsim_rank_in_topk': int(gold_pos[0]) + 1 if len(gold_pos) else None,
        'methods': rank_methods(scores, ranked, gold), 'truncation': totals,
    }

In [ ]:
def save_json(data, path):
    with TemporaryDirectory(dir=path.parent) as tmp:
        staged = Path(tmp) / path.name
        staged.write_text(json.dumps(data, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        staged.replace(path)


def save_csv(frame, path):
    with TemporaryDirectory(dir=path.parent) as tmp:
        staged = Path(tmp) / path.name
        frame.to_csv(staged, index=False)
        staged.replace(path)


spec = {
    'format_version': 1, 'source_run': SOURCE_RUN.name, 'cross_sha256': file_sha(CROSS_WEIGHTS),
    'colbert_epoch': COLBERT_EPOCH, 'cross_epoch': CROSS_EPOCH, 'mining': MINING,
    'val_query_ids': VAL_IDS, 'source_val_set': SAVED['val_set'],
    'candidate_sha256': hashlib.sha256(CANDIDATES.tobytes() + ANN_RANKS.tobytes()).hexdigest(),
    'backbone': backbone, 'tokenizer_sha256': tokenizer_hash,
    'cross_tokens': TRAIN_LENGTH, 'cross_tokens_per_review': PER_REVIEW, 'truncation': 'longest_first',
    'split_review_numbers': [[1, 2, 3], [4, 5]], 'group_weights': [0.5, 0.5], 'lse_temperature': 1.0,
    'embedding': 'last_hidden_state[:, 0, :] before pre_classifier', 'methods': list(METHODS),
    'cross_infer_batch': BATCH, 'device': DEVICE.type, 'precision': 'bfloat16' if USE_BF16 else 'float32',
    'torch_version': str(torch.__version__), 'transformers_version': str(transformers.__version__),
}
report = RUN_DIR / 'cross_review_split_validation' / SOURCE_RUN.name / signature(spec)[:16]
shards = report / 'parts'
shards.mkdir(parents=True, exist_ok=True)
save_json({**spec, 'cross_weights': str(CROSS_WEIGHTS)}, report / 'settings.json')

records = []
chunk_size = 10  # 중단되어도 완료한 10개 단위는 재사용한다.
with tqdm(total=len(VAL), desc='VAL queries') as progress:
    for start in range(0, len(VAL), chunk_size):
        end = min(start + chunk_size, len(VAL))
        path = shards / f'{start:04d}_{end:04d}.json'
        if path.is_file():
            part = read_json(path)
            if [r['query_id'] for r in part] != [str(qid) for qid in VAL_IDS[start:end]]:
                raise ValueError(f'저장한 결과의 VAL 순서가 다릅니다. {path}')
            progress.update(len(part))
        else:
            part = []
            for i in range(start, end):
                part.append(score_query(i))
                progress.update(1)
            save_json(part, path)
        records.extend(part)

## 4. 비교표 읽기

`stage_summary.csv`의 **P@1**, **R@10**, **MRR@100**은 전체 VAL 기준이다. 정답이 후보에 없으면 세 지표 모두 그 쿼리에 0을 준다. **`cross_top1_given_candidate`**는 정답이 100개 후보 안에 있을 때의 1위 정답률이다.

**`top1_gained` / `top1_lost`**는 Full 5 대비 각각 새로 맞힌 쿼리와 놓친 쿼리 수다. **`delta_p_at_1_pp`**는 P@1의 퍼센트포인트 차이다. 이전 448토큰 실험에서는 350개 중 253개가 후보에 들어왔고 99개가 1위였다. 아래 표의 차이는 이번 실행의 Full 5 행을 기준으로 계산한다.

`query_diagnostics.csv`에는 쿼리·방식별 정답 순위와 1위 상품 ID가 있다. `truncation_summary.csv`에서 `review_side_trunc_rate`는 입력 쌍 중 리뷰 쪽이 448토큰 상한으로 잘린 비율이다. `review_cap_rate`는 원문 리뷰 중 58토큰을 넘긴 비율이다. 상품 메타데이터는 A와 B에 각각 들어가므로 보존된 상품 토큰 수는 입력 쌍당 평균으로 비교한다.

In [ ]:
diagnostics = []
token_rows = []
for rec in records:
    common = {key: rec[key] for key in ('query_id', 'gold_item_id', 'gold_ann_rank_in_pool', 'gold_maxsim_rank_in_topk')}
    for method in METHODS:
        diagnostics.append({**common, 'method': method, **rec['methods'][method]})
    for group in REVIEW_GROUPS:
        token_rows.append({'review_group': group, **rec['truncation'][group]})

detail = pd.DataFrame(diagnostics)
for col in ('gold_ann_rank_in_pool', 'gold_maxsim_rank_in_topk', 'cross_gold_rank'):
    detail[col] = detail[col].astype('Int64')
ranks = detail['cross_gold_rank']
detail['cross_top1_hit'] = ranks.eq(1).fillna(False).astype(bool)
detail['cross_top10_hit'] = ranks.le(10).fillna(False).astype(bool)
detail['reciprocal_rank'] = (1 / ranks.astype('Float64')).fillna(0.0).astype(float)
base = detail.loc[detail['method'] == 'Full 5'].set_index('query_id', verify_integrity=True)
detail['full5_gold_rank'] = detail['query_id'].map(base['cross_gold_rank']).astype('Int64')
detail['full5_top1_hit'] = detail['query_id'].map(base['cross_top1_hit'])

summary_rows = []
for method in METHODS:
    frame = detail.loc[detail['method'] == method]
    hit = frame['cross_top1_hit']
    base_hit = frame['full5_top1_hit']
    n = len(frame)
    in_candidates = int(frame['gold_maxsim_rank_in_topk'].notna().sum())
    gained, lost = int((hit & ~base_hit).sum()), int((~hit & base_hit).sum())
    summary_rows.append({
        'method': method, 'n_queries': n, 'colbert_epoch': COLBERT_EPOCH, 'cross_epoch': CROSS_EPOCH,
        'cross_tokens': TRAIN_LENGTH, 'cross_tokens_per_review': PER_REVIEW, **SEARCH,
        'ann_recall_at_900': frame['gold_ann_rank_in_pool'].notna().mean(),
        'colbert_hits': in_candidates, 'colbert_recall_at_100': in_candidates / n,
        'top1_hits': int(hit.sum()), 'cross_precision_at_1': hit.mean(),
        'cross_recall_at_10': frame['cross_top10_hit'].mean(),
        'cross_mrr_at_100': frame['reciprocal_rank'].mean(),
        'cross_top1_given_candidate': hit.sum() / in_candidates if in_candidates else np.nan,
        'top1_gained': gained, 'top1_lost': lost, 'delta_p_at_1_pp': 100 * (gained - lost) / n,
        'delta_r_at_10_pp': 100 * (frame['cross_top10_hit'].mean() - base['cross_top10_hit'].mean()),
    })
summary = pd.DataFrame(summary_rows)

trunc = pd.DataFrame(token_rows).groupby('review_group', sort=False).sum()
trunc['review_cap_rate'] = trunc['n_capped'] / trunc['n_reviews']
trunc['pair_trunc_rate'] = trunc['truncated_pairs'] / trunc['n_pairs']
trunc['review_side_trunc_rate'] = trunc['review_side_truncated_pairs'] / trunc['n_pairs']
trunc['metadata_side_trunc_rate'] = trunc['metadata_side_truncated_pairs'] / trunc['n_pairs']
for col in ('review_tokens_removed', 'metadata_tokens_removed', 'metadata_tokens_kept'):
    trunc['mean_' + col] = trunc[col] / trunc['n_pairs']
trunc = trunc.reset_index()

save_csv(summary, report / 'stage_summary.csv')
save_csv(detail, report / 'query_diagnostics.csv')
save_csv(trunc, report / 'truncation_summary.csv')
display(summary[['method', 'top1_hits', 'cross_precision_at_1', 'cross_recall_at_10', 'cross_mrr_at_100',
                 'delta_p_at_1_pp', 'top1_gained', 'top1_lost', 'cross_top1_given_candidate']].round(4))
display(trunc[['review_group', 'n_pairs', 'review_cap_rate', 'review_side_trunc_rate', 'metadata_side_trunc_rate',
               'mean_review_tokens_removed', 'mean_metadata_tokens_kept']].round(4))
removed = trunc.set_index('review_group')['mean_review_tokens_removed']
print(f"448토큰 상한으로 잘린 리뷰 쪽 토큰 (후보당 평균) | Full 5 {removed['1-5']:.1f}"
      f" | 3+2 합계 {removed['1-3'] + removed['4-5']:.1f}")
print('결과 폴더', report)